In [0]:
# Databricks notebook source
import sys
import os
from pyspark.sql.functions import lit

# --- Parameter Retrieval (Environment Layer) ---
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("raw_path", "") 
dbutils.widgets.text("job_run_id", "") 

CATALOG = dbutils.widgets.get("catalog_name") 
RAW_PATH = dbutils.widgets.get("raw_path")
JOB_RUN_ID = dbutils.widgets.get("job_run_id")

if not CATALOG or not RAW_PATH:
    raise ValueError("CATALOG or RAW_PATH parameter is strictly required by the parent job.")

# --- Dynamic Root Discovery (Pathing Layer) ---
project_root = os.getcwd()
while project_root != '/' and not os.path.exists(os.path.join(project_root, 'src')):
    project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- Explicit Library Imports (Modularity Layer) ---
from src.config.paths import ProjectConfig
from src.config.schemas import POINTS_MAPPING_SCHEMA
from src.utils.spark_io import read_reference_source, write_reference_table
from src.utils.audit import get_batch_id, add_ref_metadata

# Instantiate Configuration
cfg = ProjectConfig(catalog=CATALOG, raw_path=RAW_PATH)

In [0]:
# --- Pipeline Execution (Logic Layer) ---
batch_id = get_batch_id(JOB_RUN_ID)
print(f"Executing Customer Ingestion \nCatalog: {cfg.CATALOG} \nBatch ID: {batch_id}")

from pyspark.sql.functions import lit

# 1. Execute the read logic
df_points = read_reference_source(
    spark=spark, 
    source_path=cfg.POINTS_SOURCE_PATH, 
    schema=POINTS_MAPPING_SCHEMA
).withColumn("_batch_id", lit(batch_id))

# 2. Enrich with Silver Metadata
df_points_final = add_ref_metadata(df_points)

# 4. Execute the write
write_reference_table(
    df=df_points_final, 
    target_table=cfg.REF_SCORE_POINTS
)

In [0]:
# %sql
# select * from lending_risk_dev.silver.ref_score_points_mapping;